# Remap open-vocab labels → fixed YOLO `nc`

Use after **`qwen_sam3_image_pipeline.ipynb`** with **`EXPORT_OPEN_VOCAB_IDS = True`**.

Reads each run’s **`labels/metadata.json`** (`label_export_mode: open_vocab`) and **`labels/segments/*.txt`** (class column = open-vocab id). Rewrites **segment `.txt`**, **`labels/segments/classes.txt`**, and **`metadata.json`** using **`TARGET_YOLO_CLASS_NAMES`** + **`TARGET_YOLO_CLASS_ALIASES`** (same resolver idea as the image pipeline).

Optional backup of segment files to **`labels/segments_open_vocab_orig/`** before overwrite.

**Related:** **`standardize_open_vocab_extend_yolo.ipynb`** — if you prefer a **large extended** class list (standardized Qwen strings as `names`) instead of collapsing into a small taxonomy, use that notebook instead of this one.

In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

# Root like train_yolo_seg / mine_taxonomy (folders with labels/metadata.json under each image stem)
ARTIFACTS_ROOT = Path("/Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/outputs/uss_pf")

TARGET_YOLO_CLASS_NAMES: list[str] = [
    "road",
    "pavement",
    "building",
    "wall",
    "car",
    "tree",
    "sky",
    "person",
    "fence",
    "pole",
    "sign",
    "barrier",
    "railing",
    "overhead",
    "cone",
    "structure",
]
TARGET_YOLO_CLASS_ALIASES: dict[str, str] = {
    "automobile": "car",
    "suv": "car",
    "pedestrian": "person",
    "cars": "car",
    "cars parked": "car",
    "red car": "car",
    "black car": "car",
    "black suv": "car",
    "black SUV": "car",
    "minivan": "car",
    "white van": "car",
    "truck": "car",
    "volkswagen car": "car",
    "Volkswagen car": "car",
    "roadway": "road",
    "road surface": "road",
    "concrete road": "road",
    "concrete road surface": "road",
    "cobbled street": "road",
    "parking lot": "pavement",
    "concrete parking lot": "pavement",
    "brick pavement": "pavement",
    "concrete pavement": "pavement",
    "sidewalk": "pavement",
    "concrete floor": "pavement",
    "buildings": "building",
    "building exterior": "building",
    "building with green roof": "building",
    "rooftops": "building",
    "red roof": "building",
    "door": "building",
    "window": "building",
    "window blinds": "building",
    "concrete wall": "wall",
    "building wall": "wall",
    "trees": "tree",
    "trees in the distance": "tree",
    "hedge": "tree",
    "tree planter": "tree",
    "concrete barrier": "barrier",
    "red and white barrier": "barrier",
    "yellow poles": "pole",
    "street signs": "sign",
    "road signs": "sign",
    "signpost": "sign",
    "signboard": "sign",
    "advertising sign": "sign",
    "overhead sign": "sign",
    "license plate": "sign",
    "overhead structure": "overhead",
    "ceiling": "overhead",
    "ceiling beams": "overhead",
    "metal beams": "overhead",
    "overhead bridge": "overhead",
    "concrete overpass": "overhead",
    "carport": "overhead",
    "overhead canopy": "overhead",
    "traffic cone": "cone",
    "yellow cone": "cone",
    "metal railing": "railing",
    "metal staircase": "structure",
    "concrete pillars": "structure",
    "concrete planters": "structure",
    "concrete blocks": "structure",
    "metal pipes": "structure",
    "pipes": "structure",
    "white box": "structure",
}

SKIP_UNMAPPED_LINES = True  # drop segment rows whose open-vocab name does not resolve
BACKUP_OPEN_VOCAB_SEGMENTS = True  # copy labels/segments/*.txt to labels/segments_open_vocab_orig/
DRY_RUN = False

In [ ]:
def resolve_yolo_class_id(prompt: str, names: list[str], aliases: dict[str, str]) -> int | None:
    raw = prompt.strip()
    lk = raw.lower()
    mapped = None
    for ak, av in aliases.items():
        if ak.lower() == lk:
            mapped = av
            break
    candidate = (mapped if mapped is not None else raw).strip()
    cl = candidate.lower()
    for i, name in enumerate(names):
        if name.lower() == cl:
            return i
    return None


def iter_image_run_dirs(artifacts_root: Path):
    """Flat runs (stem/) or nested (seq/stem/) with labels/metadata.json."""
    if not artifacts_root.is_dir():
        raise FileNotFoundError(artifacts_root)
    for sub in sorted(artifacts_root.iterdir()):
        if not sub.is_dir() or sub.name.startswith("."):
            continue
        meta_flat = sub / "labels" / "metadata.json"
        if meta_flat.is_file():
            yield sub.name, sub
            continue
        for nest in sorted(sub.iterdir()):
            if not nest.is_dir() or nest.name.startswith("."):
                continue
            if (nest / "labels" / "metadata.json").is_file():
                yield f"{sub.name}__{nest.name}", nest


def remap_segment_file(
    seg_txt: Path,
    open_names: list[str],
    names: list[str],
    aliases: dict[str, str],
    *,
    skip_unmapped: bool,
) -> tuple[list[str], int, list[str]]:
    """Returns (new_lines, dropped_unmapped, issues)."""
    raw = seg_txt.read_text().strip()
    if not raw:
        return [], 0, []
    out_lines: list[str] = []
    issues: list[str] = []
    dropped = 0
    for li, line in enumerate(raw.splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        try:
            old_cid = int(float(parts[0]))
        except ValueError:
            issues.append(f"{seg_txt.name} L{li}: bad class token")
            continue
        if old_cid < 0 or old_cid >= len(open_names):
            issues.append(f"{seg_txt.name} L{li}: class {old_cid} out of range for open vocab")
            continue
        pname = open_names[old_cid]
        new_cid = resolve_yolo_class_id(pname, names, aliases)
        if new_cid is None:
            dropped += 1
            if not skip_unmapped:
                issues.append(f"{seg_txt.name} L{li}: unmapped open name {pname!r}")
            continue
        out_lines.append(f"{new_cid} " + " ".join(parts[1:]))
    return out_lines, dropped, issues


print("Helpers OK")

In [ ]:
runs_ok = 0
runs_skip = 0
runs_err = 0
total_kept = 0
total_drop = 0
all_issues: list[str] = []

for key, run_dir in iter_image_run_dirs(ARTIFACTS_ROOT):
    meta_path = run_dir / "labels" / "metadata.json"
    seg_dir = run_dir / "labels" / "segments"
    try:
        meta = json.loads(meta_path.read_text())
    except Exception as e:
        runs_err += 1
        all_issues.append(f"{key}: metadata read error: {e}")
        continue
    if meta.get("label_export_mode") != "open_vocab":
        runs_skip += 1
        continue
    open_names = meta.get("yolo_class_names") or []
    if not isinstance(open_names, list) or not open_names:
        runs_err += 1
        all_issues.append(f"{key}: missing yolo_class_names in open-vocab metadata")
        continue

    label_txts = [p for p in seg_dir.glob("*.txt") if p.name != "classes.txt"]
    if not label_txts:
        runs_err += 1
        all_issues.append(f"{key}: no segment label txt")
        continue

    backup_dir = run_dir / "labels" / "segments_open_vocab_orig"
    if BACKUP_OPEN_VOCAB_SEGMENTS and not DRY_RUN:
        backup_dir.mkdir(parents=True, exist_ok=True)
        for p in label_txts:
            shutil.copy2(p, backup_dir / p.name)

    run_issues: list[str] = []
    rk = rd = 0
    for seg_txt in sorted(label_txts):
        new_lines, d, iss = remap_segment_file(
            seg_txt,
            open_names,
            TARGET_YOLO_CLASS_NAMES,
            TARGET_YOLO_CLASS_ALIASES,
            skip_unmapped=SKIP_UNMAPPED_LINES,
        )
        rk += len(new_lines)
        rd += d
        run_issues.extend(iss)
        if not DRY_RUN:
            seg_txt.write_text("\n".join(new_lines))

    if not DRY_RUN:
        cls_txt = "\n".join(TARGET_YOLO_CLASS_NAMES)
        (seg_dir / "classes.txt").write_text(cls_txt)
        meta["label_export_mode"] = "yolo_closed"
        meta["yolo_nc"] = len(TARGET_YOLO_CLASS_NAMES)
        meta["yolo_class_names"] = list(TARGET_YOLO_CLASS_NAMES)
        meta["yolo_class_aliases"] = dict(TARGET_YOLO_CLASS_ALIASES)
        meta["remapped_from_open_vocab"] = True
        meta["open_vocab_class_names_snapshot"] = list(open_names)
        meta_path.write_text(json.dumps(meta, indent=2))

    total_kept += rk
    total_drop += rd
    all_issues.extend(run_issues)
    runs_ok += 1
    print(f"{'[dry] ' if DRY_RUN else ''}{key}: kept_lines={rk} dropped={rd} issues={len(run_issues)}")

print(f"\nDone. runs_remapped={runs_ok} runs_skipped_not_open_vocab={runs_skip} runs_err={runs_err}")
print(f"total segment rows kept (est)={total_kept} dropped(unmapped)={total_drop}")
if all_issues:
    print("\nIssues (first 20):")
    for s in all_issues[:20]:
        print("  ", s)